**Prediksi Modern & Machine Learning**

Alice Delica Permata R.

2404220017

**EXPERIMENT 0: DATA UNDERSTANDING & PREPARATION**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
df = pd.read_csv('UCI_Credit_Card.csv')

Dimensi, struktur, dan tipe data

In [ ]:
print("Dimensi Data:", df.shape)
print("\nInfo Data:")
print(df.info())

Dimensi Data: (30000, 25)

Info Data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ID                          30000 non-null  int64  
 1   LIMIT_BAL                   30000 non-null  float64
 2   SEX                         30000 non-null  int64  
 3   EDUCATION                   30000 non-null  int64  
 4   MARRIAGE                    30000 non-null  int64  
 5   AGE                         30000 non-null  int64  
 6   PAY_0                       30000 non-null  int64  
 7   PAY_2                       30000 non-null  int64  
 8   PAY_3                       30000 non-null  int64  
 9   PAY_4                       30000 non-null  int64  
 10  PAY_5                       30000 non-null  int64  
 11  PAY_6                       30000 non-null  int64  
 12  BILL_AMT1                   30000 non-null  float6

 Dataset terdiri dari 30.000 baris observasi dan 25 kolom.   

 Seluruh variabel bertipe numerik, terdiri atas 13 kolom bertipe desimal float64 seperti limit kredit serta nominal tagihan dan pembayaran, dan 12 kolom bertipe bilangan bulat int64 seperti identitas, usia, status riwayat pembayaran, serta variabel target.

Identifikasi target, missing values, duplicate

In [ ]:
print("\nJumlah Missing Value per Kolom:\n", df.isnull().sum())
print("Jumlah Baris Duplikat:", df.duplicated().sum())


Jumlah Missing Value per Kolom:
 ID                            0
LIMIT_BAL                     0
SEX                           0
EDUCATION                     0
MARRIAGE                      0
AGE                           0
PAY_0                         0
PAY_2                         0
PAY_3                         0
PAY_4                         0
PAY_5                         0
PAY_6                         0
BILL_AMT1                     0
BILL_AMT2                     0
BILL_AMT3                     0
BILL_AMT4                     0
BILL_AMT5                     0
BILL_AMT6                     0
PAY_AMT1                      0
PAY_AMT2                      0
PAY_AMT3                      0
PAY_AMT4                      0
PAY_AMT5                      0
PAY_AMT6                      0
default.payment.next.month    0
dtype: int64
Jumlah Baris Duplikat: 0


Tidak adanya nilai kosong dan baris duplikat pada dataset.

Distribusi target

In [ ]:
print("\nDistribusi Target (default.payment.next.month):")
print(df['default.payment.next.month'].value_counts(normalize=True))


Distribusi Target (default.payment.next.month):
default.payment.next.month
0    0.7788
1    0.2212
Name: proportion, dtype: float64


Distribusi variabel target menunjukkan adanya ketidakseimbangan kelas (imbalanced data), di mana mayoritas nasabah berstatus tidak default (kelas 0 sekitar 77,88%) dan minoritas mengalami default (kelas 1 sekitar 22,12%). Oleh karena itu, pembagian data dilakukan menggunakan Stratified Train-Test Split dan Stratified K-Fold agar proporsi kelas tetap konsisten di setiap partisi data.   Metrik utama yang digunakan adalah F1-score, bukan sekadar akurasi, untuk mengevaluasi kemampuan model terhadap kelas minoritas secara adil.

Menentukan Predictor dan Target (disini kita mengeluarkan kolom id, agar model tidak mempelajari nomor urut data)

In [ ]:
X = df.drop(columns=['ID', 'default.payment.next.month'])
y = df['default.payment.next.month']

Train-Test Split (80:20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Meyiapkan 5-Fold Stratified CV

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Helper function untuk evaluasi metrik

In [ ]:
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te, cv_strat):
    cv_f1 = cross_val_score(model, X_tr, y_tr, cv=cv_strat, scoring='f1').mean()
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec = recall_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)

    print(f"{name}")
    print(f"CV F1-score   : {cv_f1:.4f}")
    print(f"Test Accuracy : {acc:.4f}")
    print(f"Test Precision: {prec:.4f}")
    print(f"Test Recall   : {rec:.4f}")
    print(f"Test F1-score : {f1:.4f}\n")
    return {'Model': name, 'CV F1': cv_f1, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}

results = []

**EXPERIMENT 1: BASELINE MODEL**

Logistic Regression tanpa feature scaling atau preprocessing

In [ ]:
lr_base = LogisticRegression(max_iter=1000, random_state=42)
res_exp1 = evaluate_model("Experiment 1 - Baseline LR", lr_base, X_train, y_train, X_test, y_test, cv)
results.append(res_exp1)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Experiment 1 - Baseline LR
CV F1-score   : 0.3525
Test Accuracy : 0.8037
Test Precision: 0.6630
Test Recall   : 0.2283
Test F1-score : 0.3397



/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pada model baseline tanpa penskalaan fitur, algoritma solver lbfgs memunculkan peringatan ConvergenceWarning karena batas iterasi (max_iter=1000) tercapai sebelum model mencapai konvergensi optimal. Model menghasilkan CV F1-score sebesar 0.3525 dan Test F1-score sebesar 0.3397 yang menjadi baseline untuk eksperimen berikutnya.

**EXPERIMENT 2: PREPROCESSING**

Standard scaling dimasukkan ke dalam Pipeline untuk mencegah data leakage.

In [ ]:
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])
res_exp2 = evaluate_model("Experiment 2 - Scaled LR", pipe_lr, X_train, y_train, X_test, y_test, cv)
results.append(res_exp2)

Experiment 2 - Scaled LR
CV F1-score   : 0.3647
Test Accuracy : 0.8077
Test Precision: 0.6868
Test Recall   : 0.2396
Test F1-score : 0.3553



Penerapan StandardScaler melalui Pipeline berhasil meniadakan peringatan konvergensi pada solver lbfgs. Seluruh metrik mengalami peningkatan. Penskalaan fitur terbukti memberikan dampak positif terhadap model.

**EXPERIMENT 3: MODEL COMPARISON**

Decision Tree

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
res_dt = evaluate_model("Experiment 3 - Decision Tree", dt, X_train, y_train, X_test, y_test, cv)
results.append(res_dt)

Experiment 3 - Decision Tree
CV F1-score   : 0.4020
Test Accuracy : 0.7145
Test Precision: 0.3694
Test Recall   : 0.4115
Test F1-score : 0.3893



Random Forest

In [ ]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
res_rf = evaluate_model("Experiment 3 - Random Forest", rf, X_train, y_train, X_test, y_test, cv)
results.append(res_rf)

Experiment 3 - Random Forest
CV F1-score   : 0.4808
Test Accuracy : 0.8120
Test Precision: 0.6325
Test Recall   : 0.3580
Test F1-score : 0.4572



Dari ketiga model yang dibandingkan, Random Forest menghasilkan performa terbaik dengan CV F1-score tertinggi (0.4808) dan Test F1-score tertinggi (0.4572). Decision Tree menghasilkan F1-score yang lebih rendah serta akurasi 71,45% (overfitting). Maka, Random Forest dipilih sebagai model yang akan dioptimasi pada Experiment 4.

**EXPERIMENT 4: HYPERPARAMETER TUNING**

Dari Experiment 3 didapatkan model terbaik dari adalah Random Forest.

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [8, 12],
    'class_weight': ['balanced', None],
    'min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)
best_rf = grid_rf.best_estimator_
y_pred_tune = best_rf.predict(X_test)

res_exp4 = {
    'Model': 'Experiment 4 - Tuned RF',
    'CV F1': grid_rf.best_score_,
    'Accuracy': accuracy_score(y_test, y_pred_tune),
    'Precision': precision_score(y_test, y_pred_tune, zero_division=0),
    'Recall': recall_score(y_test, y_pred_tune),
    'F1': f1_score(y_test, y_pred_tune)
}
print("Experiment 4 - Tuned Random Forest")
print("Best Params   :", grid_rf.best_params_)
print(f"CV F1-score   : {res_exp4['CV F1']:.4f}")
print(f"Test Accuracy : {res_exp4['Accuracy']:.4f}")
print(f"Test Precision: {res_exp4['Precision']:.4f}")
print(f"Test Recall   : {res_exp4['Recall']:.4f}")
print(f"Test F1-score : {res_exp4['F1']:.4f}\n")
results.append(res_exp4)

Experiment 4 - Tuned Random Forest
Best Params   : {'class_weight': 'balanced', 'max_depth': 8, 'min_samples_split': 2, 'n_estimators': 200}
CV F1-score   : 0.5448
Test Accuracy : 0.7823
Test Precision: 0.5069
Test Recall   : 0.5833
Test F1-score : 0.5424



**EXPERIMENT 5: ADVANCED MODEL**

In [ ]:
gb = GradientBoostingClassifier(random_state=42)
res_exp5 = evaluate_model("Experiment 5 - Gradient Boosting", gb, X_train, y_train, X_test, y_test, cv)
results.append(res_exp5)

Experiment 5 - Gradient Boosting
CV F1-score   : 0.4786
Test Accuracy : 0.8183
Test Precision: 0.6634
Test Recall   : 0.3625
Test F1-score : 0.4688



Model advanced Gradient Boosting menghasilkan CV F1 sebesar 0.4786 dan Test F1 sebesar 0.4688. Meskipun akurasi dan presisinya cukup tinggi, nilai F1-score model ini belum melampaui hasil Tuned Random Forest dari Experiment 4 (Test F1: 0.5424). Hal ini menegaskan bahwa arsitektur yang lebih kompleks tidak selalu menjamin peningkatan performa pada metrik utama tanpa penanganan khusus pada ketidakseimbangan kelas.

**SUMMARY TABLE**

In [ ]:
summary_df = pd.DataFrame(results)
print("RANGKUMAN HASIL EKSPERIMEN")
print(summary_df.to_string(index=False))

RANGKUMAN HASIL EKSPERIMEN
                           Model    CV F1  Accuracy  Precision   Recall       F1
      Experiment 1 - Baseline LR 0.352464  0.803667   0.663020 0.228335 0.339686
        Experiment 2 - Scaled LR 0.364699  0.807667   0.686825 0.239638 0.355307
    Experiment 3 - Decision Tree 0.402027  0.714500   0.369418 0.411454 0.389305
    Experiment 3 - Random Forest 0.480764  0.812000   0.632490 0.357950 0.457170
         Experiment 4 - Tuned RF 0.544766  0.782333   0.506876 0.583271 0.542397
Experiment 5 - Gradient Boosting 0.478602  0.818333   0.663448 0.362472 0.468811
